<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Visual Wake Words (VWW) — Akida 2 Benchmark

This notebook evaluates a converted Akida VWW model and, **if an Akida 2 device is connected**, benchmarks its latency. The Akida 2 reference platform is an FPGA running at 25 MHz; a projected latency at a higher clock is also reported (cycle count is clock-independent, so the projection is exact). Power measurement is not yet available, so benchmarking is latency-only.

> **Note:** the hardware benchmark section requires a physical Akida 2 FPGA board. Without a device it will report "no hardware found" and skip — it cannot run on Colab.

## Setup

In [ ]:
import os
import numpy as np
import akida

DATA_PATH = './data/vw_coco2014_96'
MODELS_DIR = './models'
INPUT_SHAPE = (96, 96, 3)

# Which converted model to benchmark. Options produced by the training pipeline:
#   akidanet_vww_i8_w8_a8.fbz     (8-bit)
#   akidanet_vww_i8_w4_a4_qat.fbz (4-bit QAT)
MODEL_FBZ = os.path.join(MODELS_DIR, 'akidanet_vww_i8_w8_a8.fbz')

# Clocks (see vww_benchmark.py).
MEASURED_CLOCK = 25e6    # 25 MHz FPGA
PROJECTED_CLOCK = 100e6  # provisional — confirm target clock

## Load the Akida model

In [ ]:
ak_model = akida.Model(MODEL_FBZ)
ak_model.summary()

## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the benchmark below is skipped.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found — benchmark will be skipped.')
else:
    print('Akida device found:', device)

## Samples

Akida latency is activity-dependent (it exploits sparsity), so we benchmark on real inputs rather than random data.

In [ ]:
from vww_data import get_samples

NUM_SAMPLES = 1000
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)

## Latency benchmark

Full-model benchmark in both mapping modes, with measured (25 MHz) and projected latency. Cycle count is fixed for a given model + mapping, so `projected_ms = mean_inf_clk / PROJECTED_CLOCK * 1000`.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    for mm in ['Minimal', 'AllNps']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        projected_ms = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(ak_model)
        print(f'[{mm}] NPs={num_nps} passes={num_passes} '
              f'latency@25MHz={res["mean_clk_ms"]:.3f} ms '
              f'projected@100MHz={projected_ms:.3f} ms')
else:
    print('Hardware not available — skipping latency benchmark.')

## Per-layer benchmark & sparsity

Per-layer latency (Minimal mapping) plus activation sparsity, which drives Akida efficiency.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
    print('Per-layer benchmark complete.')
else:
    # Sparsity can still be computed on the software backend without hardware.
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    print('Hardware not available — skipped latency; sparsity computed on software backend.')